# p2_instruction 加练题（在线练习版）

10 道手感题，练的是 README 里练习 0、2、3 要用的基本功：JSONL 读写、validate 校验、覆盖度统计。全程只用标准库 json 加 pandas，Colab 打开就能跑。

做题节奏：先预测或先报思路，再运行，再展开答案。

在线练习：

https://colab.research.google.com/github/osotoo/AI-Stack/blob/main/p2_dataset/p2_instruction/questions.ipynb

In [ ]:
import json
import pandas as pd
print("环境就绪")

## 练习输入：5 条故意埋坏的指令对

先运行下面这格，后面几题都用它。哪几行是坏的，先自己肉眼找一遍。

In [ ]:
pairs = [
    {"instruction": "帮我查一下订单状态", "input": "订单号 2026031102358", "output": "该订单已发货，预计 3 天内送达。", "task_type": "事实问答"},
    {"instruction": "怎么申请退款", "input": "", "output": "TODO", "task_type": "流程操作"},
    {"instruction": "", "input": "", "output": "请提供订单号后我来帮您查询物流状态。", "task_type": "事实问答"},
    {"instruction": "快递太慢了怎么投诉", "input": "", "output": "转人工", "task_type": "话术生成"},
    {"instruction": "帮我改收货地址", "input": "新地址：杭州市某区某路 1 号", "output": "地址已更新，下个订单生效。", "task_type": "工单处理"},
]
TASK_TYPES = ["事实问答", "流程操作", "排查诊断", "话术生成"]
print(len(pairs), "条")

### 题 1：从零构造 5 条指令对

按你自己的工作领域，构造 5 条指令对（字段 instruction、input、output），打印第 2 条的 output。写之前先问自己：这条数据想让模型学会什么行为？

In [ ]:
# TODO：构造 5 条指令对

### 题 2：写成 JSONL 再读回

把题 1 的列表逐行写成 warmup.jsonl（json.dumps 加换行），再逐行 json.loads 读回，打印第 2 条的 output。

附带预测：json.dumps 不加参数时，中文会变成什么样？先猜再运行。

In [ ]:
# TODO：写 JSONL，读回，打印

### 题 3：预测题——json.load 直接读 JSONL

用 json.load(文件对象) 直接读 warmup.jsonl，会发生什么？先在纸上写预测，再运行。

In [ ]:
# TODO：先写预测注释，再运行验证

### 题 4：写 validate_one

写函数 validate_one(pair)，返回 (是否合格, 原因)。四条规则：三个核心字段非空、output 长度不小于 10、不含占位符（待补充 / TODO / xxx）、task_type 在 TASK_TYPES 白名单内。

In [ ]:
# TODO：def validate_one(pair): ...

### 题 5：批量校验

用 validate_one 检查 pairs 的 5 条，不合格的打印行号和原因。先肉眼猜几行不合格，再对程序输出。

In [ ]:
# TODO：批量校验

### 题 6：覆盖度统计

统计 pairs 里各 task_type 的条数；找出 task_type 不在白名单里的行。

In [ ]:
# TODO：覆盖度

### 题 7：DataFrame 与 JSONL 互转

用 pandas 建 5 行 DataFrame（列同 pairs），转成 JSONL 导出，再读回验证条数。提示：to_dict("records")。

In [ ]:
# TODO：DataFrame -> JSONL -> 读回

### 题 8：讨论——哪个 output 能教会模型分步排查

A：已为您转接人工客服，请稍候。
B：分步排查：1. 先确认订单号是否正确；2. 在订单页查看物流状态；3. 超过 48 小时未更新，点"联系客服"转人工；4. 仍未解决可发起投诉。
哪个能教会模型分步排查？另一个会教出什么模型？

In [ ]:
# 讨论：不必写代码，把结论写进注释

### 题 9：陷阱——instruction 和 input 的分工

"帮我查订单状态，订单号 2026031102358"全塞进 instruction 对不对？改成三元组分工正确的写法。

In [ ]:
# TODO：改正这条数据

### 题 10：综合——从语料改写指令对

从 p1_corpus 的 raw_corpus.csv 挑 3 条语料，改写成 3 条指令对（instruction 针对语料提任务，input 放原文，output 写期望回答），逐条过 validate_one。这题连起了两层：清洗的产出流进构造。

In [ ]:
# TODO：3 条指令对 + validate_one

<details><summary><b>答案</b></summary>

1. 开放题。要点：instruction 说任务，input 放材料（可为空串），output 写成期望模型学会的样子。示范：{"instruction": "帮我查一下订单状态", "input": "订单号 2026031102358", "output": "该订单已发货，预计 3 天内送达。"}
2. 默认 json.dumps 把中文转成 \uXXXX 转义；加 ensure_ascii=False 保留中文。两种读回结果等价，但训练数据建议 ensure_ascii=False，肉眼可查
3. TypeError: Extra data。JSONL 每行是独立 JSON，合在一起不是合法的单个 JSON 对象，必须逐行 json.loads
4. 参考写法：依次判三个字段非空（pd.isna 或 == ""）、len(output) >= 10、占位符 in output、task_type in TASK_TYPES；收集所有原因一起返回
5. 4 行不合格：行 2 占位符 TODO、行 3 instruction 为空、行 4 output 长度 3 小于 10、行 5 task_type 不在白名单。只有行 1 合格
6. 事实问答 2、流程操作 1、话术生成 1、工单处理 1；白名单外 1 条（工单处理）
7. 条数 5，读回一致。关键一步：pd.DataFrame(pairs) 后 df.to_json("out.jsonl", orient="records", lines=True, force_ascii=False) 也可以，或 to_dict("records") 加 json
8. B。A 教会模型"一句话敷衍转人工"；B 里的分步过程才会让模型学会分步排查——这就是 README 里 CoT 结论的落地
9. 不对。instruction="帮我查订单状态"，input="订单号 2026031102358"。材料归 input，instruction 只说要干什么
10. 开放题。示范：instruction="把这条客户反馈改写成一句话工单摘要"，input=（语料原文），output=（一句 20 字以内的摘要）

</details>